# T4A — Vision domain shift (single-layer / ResNet)

**Lemma D4** · `nuisance="domain_shift"` · [Task doc](../../docs/tasks/t04a-vision-domain.md) · FINAL: `paper_code/T4/Task4A/FINAL.md`

> E1_multiscale Gram PMH on DomainNet real→sketch: **42.15%** test acc (+3.31 pp vs B0 38.84%).

| § | What you do |
|---|-------------|
| 1–4 | Install → load demo → `check_applicability` |
| 5–6 | Estimate $\Sigma_{\text{task}}$ → PMH train → Step 5 on deploy holdout |
| 7–8 | Reproduce paper scripts → plug in your data |


**Demo note:** Runnable demo = single-hook D4 with class-aligned Gram on synthetic tabular shift. Paper DomainNet multiscale — §7 scripts.


## 1 — Install


In [ ]:
!pip install -q matching-pmh torch


## 2 — Config & imports


In [ ]:
import os
import torch
from pmh.benchmark.presets import get_preset
from pmh.pytorch_eval import (
    pytorch_demo_loaders,
    pytorch_isotropic_demo_loaders,
    pytorch_multilayer_vision_demo_loaders,
    pytorch_sequence_demo_loaders,
)
from pmh import PMHConfig, PMHTrainer, evaluate_robust_fit, check_applicability, suggest_nuisance
from pmh.adoption import RECIPE_ONE_LINER, format_recipe_banner

QUICK = os.environ.get("PMH_QUICK", "").lower() in ("1", "true", "yes")
EPOCHS = 2 if QUICK else 6
SEED = 0
print(RECIPE_ONE_LINER)


## 3 — Load demo data


In [ ]:
preset = get_preset("t4_domain_d4")
N = 200 if QUICK else 500
bundle = pytorch_demo_loaders(n=N, batch_size=32, seed=SEED)
model = bundle.model
hook, head = bundle.encoder, bundle.head
train_loader, src_loader, tgt_loader, val_loader = (
    bundle.train_loader, bundle.source_batches, bundle.target_batches, bundle.val_loader,
)


## 4 — Scope (applicability)


In [ ]:
from pmh import check_applicability, suggest_nuisance

print(suggest_nuisance(has_source_labels=True, has_target_domain=True))
app = check_applicability(stack="pytorch", has_target_domain=True)
print(app.summary())
print("suggested nuisance:", app.suggested_nuisance, "(expect 'domain_shift')")


## 5 — Estimate $\Sigma_{\text{task}}$ + PMH train


In [ ]:
import copy
from pmh import PMHTrainer, PMHConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = copy.deepcopy(model).to(device)
trainer = PMHTrainer(
    m, hook=m.enc, head=m.head, nuisance="domain_shift", rank=preset.default_rank, pmh_config=preset.pmh_config, device=device,
)
trainer.fit(train_loader, source_batches=src_loader, target_batches=tgt_loader, epochs=EPOCHS)
print("preflight", trainer.artifact_.preflight, "method", getattr(trainer.artifact_, "method", None))


## 6 — Step 5 (deploy holdout)


In [ ]:
from pmh import evaluate_robust_fit

report = evaluate_robust_fit(
    m, train_loader, val_loader,
    source_batches=src_loader, target_batches=tgt_loader,
    hook=m.enc, head=m.head, nuisance="domain_shift", rank=preset.default_rank, 
    pmh_config=preset.pmh_config, epochs=max(2, EPOCHS - 2), include_falsification=True, seed=SEED,
)
print(report.summary())
if hasattr(report, "baseline_metric"):
    print("deploy holdout — baseline:", report.baseline_metric, "pmh:", report.pmh_metric)


## 7 — Paper reproduction


Frozen results: `paper_code/T4/Task4A/FINAL.md`

- **DomainNet real→sketch:** `python paper_code/T4/Task4A/run_pipeline.py`
- **Per-layer TDI geometry:** `python paper_code/T4/Task4A/tdi.py`
- **B0 / E1 / E1_multiscale training:** `python paper_code/T4/Task4A/train.py`


## 8 — Your pipeline


Swap demo loaders for your `train_loader`, `source_batches`, `target_batches`, and deploy holdout. Hook the backbone before your task head.
